# Cluster-3 Region-DAG A30 validation

**Goal.** Reproduce the controlled BF16 comparison between complete Region-DAG recomputation and conservative cached replay, then run a gated production HTTP smoke. This notebook is intentionally unexecuted in git; retain its generated JSONL, summaries, and server log as the A30 evidence.

## Setup

Run on one NVIDIA A30 with a persistent cache. Put any Hugging Face credential in `HF_TOKEN` in the environment—never in this notebook. The setup uses the repository's pinned evaluation environment. Set `RUN_CLUSTER3_SETUP=1` only when that environment has not already been created.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, time, urllib.request

REPO_URL = "https://github.com/Kd-26/HybridDiffusion.git"
BRANCH = "cluster3-region-dag-conservative-gdn"
CACHE_ROOT = Path(os.environ.get("HYBRID_DIFFUSION_CACHE_ROOT", "/persistent/hybrid-diffusion-cache"))
REPO_DIR = Path(os.environ.get("HYBRID_DIFFUSION_REPO", "/workspace/HybridDiffusion"))
MODEL_DIR = Path(os.environ.get("MODEL_DIR", CACHE_ROOT / "models/HybridDiffusion-2B"))
EVAL_VENV = Path(os.environ.get("EVAL_VENV", CACHE_ROOT / "venvs/hybrid-diffusion-eval"))
PYTHON_BIN = EVAL_VENV / "bin/python"
HF_BIN = EVAL_VENV / "bin/hf"
MODEL_ID = "yuchen-zhu-zyc/HybridDiffusion-2B"
for path in (CACHE_ROOT, MODEL_DIR.parent):
    path.mkdir(parents=True, exist_ok=True)
print({"repo": str(REPO_DIR), "cache": str(CACHE_ROOT), "model": str(MODEL_DIR)})

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
subprocess.run(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)
if os.environ.get("RUN_CLUSTER3_SETUP") == "1":
    setup_env = os.environ.copy()
    setup_env["HYBRID_DIFFUSION_CACHE_ROOT"] = str(CACHE_ROOT)
    subprocess.run(["bash", "scripts/setup_eval_env.sh"], cwd=REPO_DIR / "eval", env=setup_env, check=True)
assert PYTHON_BIN.is_file(), f"Missing {PYTHON_BIN}; set RUN_CLUSTER3_SETUP=1 and rerun this cell"
REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
assert len(REVISION) == 40
RESULT_ROOT = CACHE_ROOT / "results/cluster3" / REVISION
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print("Exact Cluster-3 revision:", REVISION)

In [ ]:
# Download to a stable local directory. Do not assume a particular shard name.
if not MODEL_DIR.exists() or not any(MODEL_DIR.glob("*.safetensors")):
    assert HF_BIN.is_file(), f"Missing Hugging Face CLI: {HF_BIN}"
    subprocess.run([str(HF_BIN), "download", MODEL_ID, "--local-dir", str(MODEL_DIR)], check=True)
weight_files = sorted(MODEL_DIR.glob("*.safetensors"))
index_files = sorted(MODEL_DIR.glob("*.safetensors.index.json"))
assert weight_files or index_files, f"No safetensor weights found in {MODEL_DIR}"
assert (MODEL_DIR / "config.json").is_file()
assert (MODEL_DIR / "tokenizer_config.json").is_file()
config = json.loads((MODEL_DIR / "config.json").read_text())
identity = json.dumps(config, sort_keys=True).lower() + " " + MODEL_DIR.name.lower()
assert "2b" in identity or config.get("hidden_size") == 2048, "Checkpoint is not confidently HybridDiffusion-2B"
print("Model ready:", MODEL_DIR, "weights:", [p.name for p in weight_files])

In [ ]:
hardware_code = r'''
import json, torch
assert torch.cuda.is_available(), "CUDA is unavailable"
p = torch.cuda.get_device_properties(0)
print(json.dumps({"name": p.name, "memory": p.total_memory, "cuda": torch.version.cuda, "bf16": torch.cuda.is_bf16_supported(), "count": torch.cuda.device_count()}))
'''
hardware = json.loads(subprocess.check_output([str(PYTHON_BIN), "-c", hardware_code], text=True).splitlines()[-1])
assert hardware["count"] >= 1 and "A30" in hardware["name"].upper(), hardware
assert hardware["bf16"], hardware
print(hardware)

## Controlled profiles

The helper writes into `results/cluster3/<exact SHA>/<profile>`. It rejects missing/duplicate records and any failed case. Launch blocking and stage synchronization are enabled only for `one1`.

In [ ]:
def verify_profile(profile):
    directory = RESULT_ROOT / profile
    summary = json.loads((directory / "summary.json").read_text())
    records = [json.loads(line) for line in (directory / "records.jsonl").read_text().splitlines() if line.strip()]
    ids = [record["case_id"] for record in records]
    assert summary["cluster3_revision"] == REVISION
    assert summary["strict_pass"] and summary["checks"]["one_record_per_case"]
    assert len(ids) == len(set(ids)) == summary["requested_cases"]
    assert all(record["case_pass"] for record in records)
    assert all(record["selected_mask_backend"] == "custom_paged" for record in records)
    return summary, records

def run_profile(profile, *, diagnostic=False, timed_repetitions=10):
    directory = RESULT_ROOT / profile
    directory.mkdir(parents=True, exist_ok=True)
    command = [str(PYTHON_BIN), "eval/scripts/cluster3_region_dag_validation.py", "--model-path", str(MODEL_DIR), "--profile", profile, "--dtype", "bfloat16", "--tp-size", "1", "--max-total-tokens", "4096", "--timed-repetitions", str(timed_repetitions), "--output-jsonl", str(directory / "records.jsonl"), "--summary-json", str(directory / "summary.json")]
    environment = os.environ.copy()
    environment["HYBRID_DIFFUSION_CACHE_ROOT"] = str(CACHE_ROOT)
    environment.pop("CUDA_LAUNCH_BLOCKING", None)
    if diagnostic:
        environment["CUDA_LAUNCH_BLOCKING"] = "1"
        command.append("--debug-sync-stages")
    subprocess.run(command, cwd=REPO_DIR, env=environment, check=True)
    return verify_profile(profile)

In [ ]:
one1_summary, one1_records = run_profile("one1", diagnostic=True)
assert one1_summary["requested_cases"] == 1
one1_summary

In [ ]:
smoke_summary, smoke_records = run_profile("smoke16")
assert smoke_summary["requested_cases"] == 16
assert any(record["expected_gdn_replay_start"] == 0 for record in smoke_records)
smoke_summary

In [ ]:
paper_summary, paper_records = run_profile("paper100")
assert paper_summary["requested_cases"] == 100
assert len({json.dumps(record["region_contract"], sort_keys=True) for record in paper_records}) == 100
paper_summary

In [ ]:
effectiveness_summary, effectiveness_records = run_profile("effectiveness", timed_repetitions=10)
assert effectiveness_summary["requested_cases"] == 9
required = {"reference_full_ms", "cached_total_ms", "full_attention_ms", "gdn_replay_ms", "mask_build_ms", "gather_scatter_ms", "cache_lookup_restore_ms"}
for record in effectiveness_records:
    assert set(record["component_timings_ms"]) == required
    for evidence in record["component_timings_ms"].values():
        assert len(evidence["samples"]) >= 10
        assert {"median", "mad", "bootstrap_ci_95"} <= set(evidence)
effectiveness_summary

## Gated production HTTP smoke

Run this only after `one1` and `smoke16` pass. The smoke submits an explicit reference-mode Region-DAG contract, waits for exactly one non-streaming response, checks that the public load endpoint returns to zero, and rejects CUDA/lifecycle/stale/fallback/recovery errors in the server log. It does not turn a reference initialization into a cached success.

In [ ]:
assert verify_profile("one1")[0]["strict_pass"]
assert verify_profile("smoke16")[0]["strict_pass"]
PORT = int(os.environ.get("CLUSTER3_SERVER_PORT", "30000"))
SERVER_LOG = RESULT_ROOT / "production_http_smoke.log"
server_env = os.environ.copy()
server_env.update({"HYBRID_DIFFUSION_CACHE_ROOT": str(CACHE_ROOT), "EVAL_VENV": str(EVAL_VENV), "PORT": str(PORT), "TP_SIZE": "1", "DTYPE": "bfloat16", "MAX_RUNNING_REQUESTS": "1", "MEM_FRACTION_STATIC": "0.75", "CUDA_GRAPH_BS": "1"})
log_handle = SERVER_LOG.open("w")
server = subprocess.Popen(["bash", "scripts/serve.sh", "self-spec", str(MODEL_DIR), "--", "--max-total-tokens", "4096"], cwd=REPO_DIR / "eval", env=server_env, stdout=log_handle, stderr=subprocess.STDOUT, text=True)

def get_json(path):
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}{path}", timeout=10) as response:
        return response.status, json.loads(response.read())

def token_hash(values):
    digest = hashlib.sha256()
    for value in values:
        digest.update(int(value).to_bytes(8, "little", signed=True))
    return digest.hexdigest()

def position_hash(start, end):
    digest = hashlib.sha256()
    digest.update(int(start).to_bytes(8, "little", signed=True))
    digest.update(int(end).to_bytes(8, "little", signed=True))
    return digest.hexdigest()

try:
    deadline = time.time() + 600
    while True:
        if server.poll() is not None:
            raise RuntimeError(f"server exited early; inspect {SERVER_LOG}")
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=10) as health_response:
                status = health_response.status
            if status == 200:
                break
        except Exception:
            pass
        if time.time() >= deadline:
            raise TimeoutError("server health timeout")
        time.sleep(2)
    tokenizer_code = "from transformers import AutoTokenizer; import json,sys; t=AutoTokenizer.from_pretrained(sys.argv[1], trust_remote_code=True); print(json.dumps(t.encode('Explain why exact cache identity matters.', add_special_tokens=True)))"
    prompt_ids = json.loads(subprocess.check_output([str(PYTHON_BIN), "-c", tokenizer_code, str(MODEL_DIR)], text=True).splitlines()[-1])
    mask_id = 248077
    input_ids = prompt_ids + [mask_id] * 7
    boundary = len(prompt_ids)
    contract = {"sequence_length": len(input_ids), "diffusion_steps": 2, "attention_contract_id": "region_dag_conservative_gdn_v1", "regions": [{"region_id": "A", "region_version": 0, "start": 0, "end": boundary, "status": "stable", "parent_region_ids": [], "recorded_parent_versions": [], "token_hash": token_hash(input_ids[:boundary]), "position_hash": position_hash(0, boundary), "attention_contract_id": "region_dag_conservative_gdn_v1"}, {"region_id": "B", "region_version": 1, "start": boundary, "end": len(input_ids), "status": "active", "parent_region_ids": ["A"], "recorded_parent_versions": [["A", 0]], "token_hash": token_hash(input_ids[boundary:]), "position_hash": position_hash(boundary, len(input_ids)), "attention_contract_id": "region_dag_conservative_gdn_v1"}]}
    payload = {"input_ids": input_ids, "stream": False, "sampling_params": {"temperature": 0.0, "max_new_tokens": 1, "custom_params": {"region_dag": {"execution_spec": contract, "edited_regions": ["B"], "mode": "reference", "allow_full_replay": False}}}}
    request = urllib.request.Request(f"http://127.0.0.1:{PORT}/generate", data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(request, timeout=300) as response:
        assert response.status == 200
        generated = json.loads(response.read())
    assert generated and (generated.get("text") is not None or generated.get("output_ids") is not None), generated
    deadline = time.time() + 30
    while True:
        _, load = get_json("/get_load")
        entries = load if isinstance(load, list) else [load]
        if all(item.get("num_reqs", 0) == 0 and item.get("num_waiting_reqs", 0) == 0 for item in entries):
            break
        if time.time() >= deadline:
            raise RuntimeError(f"request-pool load did not return to zero: {load}")
        time.sleep(1)
finally:
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server.kill(); server.wait()
    log_handle.close()
log_text = SERVER_LOG.read_text(errors="replace").lower()
for forbidden in ("illegal memory access", "cuda error", "restore_miss", "stale state", "fallback", "recovery_replay"):
    assert forbidden not in log_text, forbidden
print("Production response:", generated)
print("Server log:", SERVER_LOG)

## Checks and next steps

Archive the exact-SHA result directory and notebook outputs. Report every failed case rather than rerunning only favorable layouts. Interpret effectiveness from `reference_full_ms` and `cached_total_ms` separately; the host paired timer is not a speedup measurement. If controlled validation passes but the production smoke fails, stop and report it as a production blocker—do not add fallback or recovery replay.